<a href="https://colab.research.google.com/github/rebecadomingoos/DesafiosFacu/blob/main/Projeto_Cinema_Atualizado_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Projeto Cinema atualizado.

In [ ]:
# =============================================================
# SISTEMA DE GESTÃO DE CINEMA E INGRESSOS 🍿  -  CineVibe
# Equipe: Luiz Jardel + Ana Rebeca
# =============================================================

print("====CineVibe====")

# -------------------------------------------------------------
# 1) DADOS GLOBAIS (estado mantido em memória)
# -------------------------------------------------------------
PRECOS = {"inteira": 30.0, "meia": 15.0, "vip": 45.0}
LETRAS = "ABCDEFGHIJ"          # até 10 fileiras
POR_FILEIRA = 10               # 10 assentos por fileira -> capacidade máx. 100

sessoes = []


In [ ]:
# -------------------------------------------------------------
# 2) FUNÇÕES AUXILIARES
# -------------------------------------------------------------
def gerar_assentos_por_capacidade(capacidade, por_fileira=POR_FILEIRA):
    """Gera os nomes dos assentos (A1, A2, ... B1, B2, ...)."""
    lista_assentos = []
    for i in range(capacidade):
        fileira = LETRAS[i // por_fileira]
        numero = i % por_fileira + 1
        lista_assentos.append(fileira + str(numero))
    return lista_assentos


def criar_sessao(id_sessao, filme, horario, sala, assentos):
    """Monta o dicionário de uma sessão."""
    return {
        "id_sessao": id_sessao,
        "filme": filme,
        "horario": horario,
        "sala": sala,
        "assentos": assentos,
        "ingressos": [],
    }


def proximo_id(lista):
    """Próximo ID disponível (maior ID existente + 1)."""
    if len(lista) == 0:
        return 1
    maior = 0
    for s in lista:
        if s["id_sessao"] > maior:
            maior = s["id_sessao"]
    return maior + 1


def assentos_ocupados(sessao):
    """Lista de assentos que já têm ingresso vendido."""
    ocupados = []
    for ingresso in sessao["ingressos"]:
        ocupados.append(ingresso["assento"])
    return ocupados


def assentos_livres(sessao):
    """Calculado na hora (assentos - ocupados), nunca guardado:
    assim os dados não ficam inconsistentes."""
    ocupados = assentos_ocupados(sessao)
    livres = []
    for assento in sessao["assentos"]:
        if assento not in ocupados:
            livres.append(assento)
    return livres


def arrecadacao_sessao(sessao):
    """Arrecadação da sessão: soma o preço de todos os ingressos vendidos."""
    total = 0.0
    for ingresso in sessao["ingressos"]:
        total += ingresso["preco"]
    return total


def ler_inteiro(mensagem):
    """Lê um inteiro maior que zero, repetindo até ser válido."""
    while True:
        texto = input(mensagem).strip()
        if texto.isdigit() and int(texto) > 0:
            return int(texto)
        print("Entrada inválida. Digite um número inteiro maior que zero.")


def ler_texto(mensagem):
    """Lê um texto não vazio, repetindo até ser válido."""
    while True:
        texto = input(mensagem).strip()
        if texto != "":
            return texto
        print("Entrada inválida. O campo não pode ficar vazio.")

In [ ]:

# -------------------------------------------------------------
# 3) SESSÕES E ASSENTOS (busca, cadastro, atualização, consulta)
# -------------------------------------------------------------
def buscar_sessao(id_sessao):
    """Busca por identificador. Retorna a sessão ou None."""
    for s in sessoes:
        if s["id_sessao"] == id_sessao:
            return s
    return None


def cadastrar_sessao(id_sessao, filme, sala, horario, capacidade):
    """Cadastra uma sessão e gera seus assentos."""
    if buscar_sessao(id_sessao) is not None:
        return False, "Já existe uma sessão com o ID " + str(id_sessao) + "."
    if filme.strip() == "" or sala.strip() == "" or horario.strip() == "":
        return False, "Filme, sala e horário não podem ficar vazios."
    if capacidade < 1 or capacidade > 100:
        return False, "Capacidade inválida (use de 1 a 100 assentos)."

    assentos = gerar_assentos_por_capacidade(capacidade)
    sessoes.append(
        criar_sessao(id_sessao, filme.strip(), horario.strip(), sala.strip(), assentos)
    )
    return True, "Sessão " + str(id_sessao) + " cadastrada com " + str(capacidade) + " assentos."


def atualizar_sessao(id_sessao, campo, novo_valor):
    """Atualiza filme, horário ou sala (ingressos vendidos são preservados)."""
    sessao = buscar_sessao(id_sessao)
    if sessao is None:
        return False, "Sessão não encontrada."
    if campo not in ("filme", "horario", "sala"):
        return False, "Campo inválido (use filme, horario ou sala)."
    if novo_valor.strip() == "":
        return False, "O novo valor não pode ser vazio."
    sessao[campo] = novo_valor.strip()
    return True, "Campo '" + campo + "' atualizado para: " + novo_valor.strip()


def listar_sessoes():
    """Mostra todas as sessões cadastradas."""
    if len(sessoes) == 0:
        print("Nenhuma sessão cadastrada.")
        return
    print("\n===== SESSÕES CADASTRADAS =====")
    for s in sessoes:
        print(
            "ID:", s["id_sessao"], "|", s["filme"], "|", s["horario"], "|", s["sala"],
            "| Livres:", len(assentos_livres(s)), "de", len(s["assentos"]),
        )


def listar_assentos(id_sessao):
    """Imprime o mapa de assentos (XX = ocupado) e devolve um resumo."""
    sessao = buscar_sessao(id_sessao)
    if sessao is None:
        return False, "Sessão não encontrada."

    ocupados = assentos_ocupados(sessao)
    print("\nSessão", sessao["id_sessao"], "-", sessao["filme"], "|",
          sessao["horario"], "|", sessao["sala"])
    print("(XX = ocupado)")

    fileira_atual = ""
    linha = ""
    for assento in sessao["assentos"]:
        if assento[0] != fileira_atual:      # mudou de fileira: imprime a anterior
            if linha != "":
                print(linha)
            fileira_atual = assento[0]
            linha = ""
        if assento in ocupados:
            linha += "XX".ljust(5)
        else:
            linha += assento.ljust(5)
    print(linha)

    livres = len(assentos_livres(sessao))
    return True, ("Livres: " + str(livres) + " | Ocupados: " + str(len(ocupados)) +
                  " | Total: " + str(len(sessao["assentos"])))


def sessao_lotada(id_sessao):
    """True se a sessão existe e não tem nenhum assento livre."""
    sessao = buscar_sessao(id_sessao)
    if sessao is None:
        return False
    return len(assentos_livres(sessao)) == 0

In [ ]:
# -------------------------------------------------------------
# 4) VENDA E CANCELAMENTO DE INGRESSOS
# -------------------------------------------------------------
def vender_ingresso(id_sessao, assento, tipo):
    """Vende um ingresso aplicando o preço do tipo.
    Impede venda em sessão lotada e venda duplicada de assento."""
    sessao = buscar_sessao(id_sessao)
    if sessao is None:
        return False, "Sessão não encontrada."

    if sessao_lotada(id_sessao):
        return False, "Ingressos lotados"

    tipo = tipo.strip().lower()
    if tipo not in PRECOS:
        return False, "Tipo inválido."

    assento = assento.strip().upper()
    if assento not in sessao["assentos"]:
        return False, "Assento " + assento + " não existe nesta sessão."

    if assento in assentos_ocupados(sessao):
        return False, "Assento " + assento + " já se encontra ocupado."

    sessao["ingressos"].append({"assento": assento, "tipo": tipo, "preco": PRECOS[tipo]})
    return True, "Ingresso vendido: " + assento + " (" + tipo + ")."


def cancelar_ingresso(id_sessao, assento):
    """Cancela o ingresso de um assento (o assento volta a ficar livre)."""
    sessao = buscar_sessao(id_sessao)
    if sessao is None:
        return False, "Sessão não encontrada."

    assento = assento.strip().upper()
    for ingresso in sessao["ingressos"]:
        if ingresso["assento"] == assento:
            sessao["ingressos"].remove(ingresso)
            return True, "Ingresso do assento " + assento + " cancelado."

    return False, "Não há ingresso vendido para o assento " + assento + "."

In [ ]:

# -------------------------------------------------------------
# 5) RELATÓRIO CONSOLIDADO (indicadores)
# -------------------------------------------------------------
def dados_gerais():
    """Relatório por sessão + 3 indicadores:
    1) valor total arrecadado; 2) ocupação geral; 3) % por tipo de ingresso."""
    if len(sessoes) == 0:
        print("Nenhuma sessão cadastrada.")
        return 0, 0.0

    total_ingressos = 0
    total_acumulado = 0.0
    total_assentos = 0
    qtd_tipos = {"inteira": 0, "meia": 0, "vip": 0}

    print("\n===== RELATÓRIO GERAL =====")

    for ses in sessoes:
        vendidos = len(ses["ingressos"])
        limite = len(ses["assentos"])
        acumulados = arrecadacao_sessao(ses)
        ocupacao = vendidos / limite * 100

        for ingr in ses["ingressos"]:
            qtd_tipos[ingr["tipo"]] += 1

        print(
            "Sessão:", ses["id_sessao"], "|",
            ses["filme"], "|",
            ses["horario"], "|",
            ses["sala"], "|",
            "Vendidos:", vendidos, "de", limite, "|",
            "Livres:", len(assentos_livres(ses)), "|",
            "Ocupação:", round(ocupacao, 1), "% |",
            "Arrecadado: R$", f"{acumulados:.2f}",
        )

        total_ingressos += vendidos
        total_acumulado += acumulados
        total_assentos += limite

    print("\n --- INDICADORES ---")
    print("1) Valor total arrecadado: R$", f"{total_acumulado:.2f}")

    ocupacao_geral = total_ingressos / total_assentos * 100
    print("2) Ocupação geral:", round(ocupacao_geral, 1), "%")

    if total_ingressos > 0:
        print("3) Inteira:", round(qtd_tipos["inteira"] / total_ingressos * 100, 1), "%")
        print("   Meia:   ", round(qtd_tipos["meia"] / total_ingressos * 100, 1), "%")
        print("   VIP:    ", round(qtd_tipos["vip"] / total_ingressos * 100, 1), "%")
    else:
        print("3) Nenhum ingresso vendido no momento.")

    return total_ingressos, total_acumulado

In [ ]:
# -------------------------------------------------------------
# 6) DADOS INICIAIS (12 registros) - estrutura inicial
# -------------------------------------------------------------
dados_iniciais = [
    # (filme, sala, horário, capacidade)
    ("Interestelar",      "Sala 1", "14:00", 12),
    ("Interestelar",      "Sala 1", "19:30", 12),
    ("Divertida Mente 2", "Sala 2", "15:00", 10),
    ("Divertida Mente 2", "Sala 2", "17:30", 10),
    ("Duna: Parte Dois",  "Sala 3", "20:00", 15),
    ("Duna: Parte Dois",  "Sala 3", "22:30", 15),
    ("Homem-Aranha",      "Sala 4", "16:00", 8),
    ("Homem-Aranha",      "Sala 4", "18:30", 8),
    ("Matrix",            "Sala 5", "21:00", 6),
    ("Coringa",           "Sala 6", "19:00", 8),
    ("Barbie",            "Sala 7", "13:30", 10),
    ("Pré-estreia",       "Sala 8", "23:00", 3),    # pequena: já nasce lotada
]
for (filme, sala, hor, cap) in dados_iniciais:
    cadastrar_sessao(proximo_id(sessoes), filme, sala, hor, cap)

# Vendas iniciais: {id da sessão: [(assento, tipo), ...]}
ingressos_iniciais = {
    1:  [("A1", "inteira"), ("A2", "meia"), ("B1", "vip")],
    2:  [("A1", "inteira"), ("A2", "inteira"), ("A3", "meia"), ("B1", "vip")],
    3:  [("A1", "meia"), ("A2", "meia")],
    5:  [("A1", "vip"), ("A2", "vip"), ("A3", "inteira")],
    7:  [("A1", "inteira"), ("A2", "meia")],
    9:  [("A1", "vip"), ("A2", "inteira")],
    12: [("A1", "inteira"), ("A2", "meia"), ("A3", "vip")],
}
for id_s in ingressos_iniciais:
    for (assento, tipo) in ingressos_iniciais[id_s]:
        vender_ingresso(id_s, assento, tipo)

print("Total de sessões:", len(sessoes))
print("Exemplo de sessão (dicionário):")
print(sessoes[0])
listar_sessoes()

In [ ]:
# -------------------------------------------------------------
# 7) TESTES (8 testes: entrada, esperado, observado)
# -------------------------------------------------------------
total_ok = 0
total_testes = 0


def testar(numero, nome, entrada, esperado, observado):
    global total_ok, total_testes
    total_testes += 1
    passou = (esperado == observado)
    if passou:
        total_ok += 1
    print("\nTeste", numero, "-", nome)
    print("   Entrada  :", entrada)
    print("   Esperado :", esperado)
    print("   Obtido   :", observado)
    print("   Resultado:", "PASSOU" if passou else "FALHOU")


print("\n===== TESTES =====")

# Teste 1: cadastrar novo registro
ok, msg = cadastrar_sessao(99, "Filme Teste", "Sala T", "10:00", 6)
testar(1, "Cadastrar nova sessão", "id 99, capacidade 6",
       (True, ["A1", "A2", "A3", "A4", "A5", "A6"]),
       (ok, buscar_sessao(99)["assentos"]))

# Teste 2: buscar existente
testar(2, "Buscar sessão existente", "id 1",
       "Interestelar", buscar_sessao(1)["filme"])

# Teste 3: buscar inexistente
testar(3, "Buscar sessão inexistente", "id 1234",
       None, buscar_sessao(1234))

# Teste 4: atualizar informação preservando consistência (caso normal de venda + update)
sessao_teste = buscar_sessao(99)
vender_ingresso(99, "A1", "inteira")
atualizar_sessao(99, "horario", "21:45")
testar(4, "Atualizar horário preservando ingressos", "id 99, horario = 21:45",
       ("21:45", 1), (sessao_teste["horario"], len(sessao_teste["ingressos"])))

# Teste 5: preço por tipo
ok, msg = vender_ingresso(99, "A2", "meia")
testar(5, "Vender ingresso aplicando preço do tipo", "id 99, A2, meia",
       (True, 15.0), (ok, sessao_teste["ingressos"][-1]["preco"]))

# Teste 6: CASO-LIMITE - assento já vendido
ok, msg = vender_ingresso(99, "A1", "vip")
testar(6, "Assento já vendido (venda duplicada)", "id 99, A1 (já ocupado), vip",
       (False, 2), (ok, len(sessao_teste["ingressos"])))

# Teste 7: CASO-LIMITE - sessão lotada
ok, msg = vender_ingresso(12, "B1", "inteira")
testar(7, "Sessão lotada", "id 12 (lotada), B1",
       (False, "Ingressos lotados", True),
       (ok, msg, sessao_lotada(12)))

# Teste 8: cancelar ingresso e conferir arrecadação (muda conforme os dados)
antes = arrecadacao_sessao(sessao_teste)       # 30 + 15 = 45
ok, msg = cancelar_ingresso(99, "A2")
testar(8, "Cancelar ingresso e recalcular arrecadação", "id 99, cancelar A2",
       (45.0, True, 30.0), (antes, ok, arrecadacao_sessao(sessao_teste)))

# Limpa a sessão de teste para não poluir os dados reais
sessoes.remove(sessao_teste)
print("\nRESULTADO FINAL:", total_ok, "de", total_testes, "testes passaram.")
print("Sessões após os testes:", len(sessoes))

In [ ]:

# -------------------------------------------------------------
# 8) MENU REPETITIVO
# -------------------------------------------------------------
def menu():
    while True:
        print("\n===== CINEVIBE - MENU =====")
        print("1 - Cadastrar sessão")
        print("2 - Buscar sessão por ID")
        print("3 - Atualizar sessão")
        print("4 - Mapa de assentos")
        print("5 - Verificar se a sessão está lotada")
        print("6 - Vender ingresso")
        print("7 - Cancelar ingresso")
        print("8 - Relatório geral")
        print("9 - Listar todas as sessões")
        print("0 - Sair")
        opcao = input("Escolha: ").strip()

        if opcao == "1":
            filme = ler_texto("Filme: ")
            sala = ler_texto("Sala: ")
            horario = ler_texto("Horário (ex.: 19:30): ")
            capacidade = ler_inteiro("Capacidade (1 a 100): ")
            ok, msg = cadastrar_sessao(proximo_id(sessoes), filme, sala, horario, capacidade)
            print(msg)

        elif opcao == "2":
            sessao = buscar_sessao(ler_inteiro("ID da sessão: "))
            if sessao is None:
                print("Sessão não encontrada.")
            else:
                print("Sessão:", sessao["id_sessao"], "|", sessao["filme"], "|",
                      sessao["horario"], "|", sessao["sala"],
                      "| Livres:", len(assentos_livres(sessao)),
                      "| Arrecadado: R$", f"{arrecadacao_sessao(sessao):.2f}")

        elif opcao == "3":
            id_s = ler_inteiro("ID da sessão: ")
            campo = ler_texto("Campo (filme/horario/sala): ").lower()
            valor = ler_texto("Novo valor: ")
            print(atualizar_sessao(id_s, campo, valor)[1])

        elif opcao == "4":
            print(listar_assentos(ler_inteiro("ID da sessão: "))[1])

        elif opcao == "5":
            id_s = ler_inteiro("ID da sessão: ")
            if buscar_sessao(id_s) is None:
                print("Sessão não encontrada.")
            elif sessao_lotada(id_s):
                print("Sessão LOTADA.")
            else:
                print("Sessão com assentos livres.")

        elif opcao == "6":
            id_s = ler_inteiro("ID da sessão: ")
            assento = ler_texto("Assento (ex.: A1): ")
            tipo = ler_texto("Tipo (inteira/meia/vip): ")
            ok, msg = vender_ingresso(id_s, assento, tipo)
            print(msg)

        elif opcao == "7":
            id_s = ler_inteiro("ID da sessão: ")
            assento = ler_texto("Assento: ")
            ok, msg = cancelar_ingresso(id_s, assento)
            print(msg)

        elif opcao == "8":
            dados_gerais()

        elif opcao == "9":
            listar_sessoes()

        elif opcao == "0":
            print("Encerrando o sistema. Até logo!")
            break

        else:
            print("Opção inválida. Escolha de 0 a 9.")


menu()